# 纯债基金久期测算系统

基于申万宏源《纯债基金久期全解析》研报的实现

## 功能说明
1. 读取基金样本数据，筛选符合条件的基金
2. 根据持仓情况表判断基金是利率债还是信用债
3. 通过Wind API获取基金后复权净值数据
4. 使用Lasso回归和带约束的WLS测算基金久期
5. 计算全市场基金久期中位数和分歧度

In [38]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import sys
import warnings
warnings.filterwarnings('ignore')

# 导入自定义模块
from data_preprocessing import FundDataPreprocessor
from fund_type_classifier import FundTypeClassifier
from wind_data_fetcher import WindDataFetcher
from bond_index_data import BondIndexDataProcessor
from duration_model import FundDurationCalculator
from data_cache import WindDataCache

print('模块导入成功')

模块导入成功


## 1. 初始化各模块

In [39]:
# 1. 数据预处理器
preprocessor = FundDataPreprocessor(
    short_term_path='短期纯债基金样本数据.xlsx',
    medium_long_term_path='中长期纯债基金样本数据.xlsx'
)

# 2. 基金类型分类器
classifier = FundTypeClassifier(
    holdings_path='纯债基金持仓情况.xlsx'
)
classifier.load_holdings_data()

# 3. Wind数据获取器（带本地缓存）
cache = WindDataCache(cache_dir='data')
wind_fetcher = WindDataFetcher(cache=cache)

# 4. 中债指数数据处理器
index_processor = BondIndexDataProcessor(
    index_path='中债财富指数.xlsx'
)
index_processor.load_price_data()
index_processor.load_duration_data()

print('所有模块初始化完成')

Wind连接成功
所有模块初始化完成


In [40]:
target_date = '2025-12-31'
weight_method = 'linear'  # 可选：'uniform' 或 'linear' 或 'exponential'

In [41]:
# ============================================================
# 数据缓存预取：检查本地缓存覆盖情况，仅对缺失数据调用Wind
# ============================================================
from WindPy import w

fund_pool_check = preprocessor.get_fund_pool(target_date)
all_fund_codes = (
    list(fund_pool_check['short']['Code']) +
    list(fund_pool_check['medium_long']['Code'])
)
print(f'基金池共 {len(all_fund_codes)} 只')

# 覆盖窗口：取90天（中长期基金最大需求），短期基金只需60天，统一用90天无损
req_start = (pd.to_datetime(target_date) - pd.Timedelta(days=90)).strftime('%Y-%m-%d')
req_end = target_date

days_actual_get = wind_fetcher.get_fund_nav(all_fund_codes[0], req_start, req_end)

# ------ 1. NAV 缓存检查 ------
missing_nav = [c for c in all_fund_codes
               if not cache.check_nav_coverage(c, days_actual_get.index[0], days_actual_get.index[-1])]

if missing_nav:
    print(f'[缓存] 需从Wind获取 {len(missing_nav)} 只基金净值（共 {len(all_fund_codes)} 只）...')
    for i, code in enumerate(missing_nav):
        df_raw = wind_fetcher.get_fund_nav(code, req_start, req_end)
        if df_raw is not None:
            cache.write_nav(code, df_raw)
        if (i + 1) % 20 == 0 or (i + 1) == len(missing_nav):
            print(f'  净值进度: {i+1}/{len(missing_nav)}')
    print('[缓存] 净值数据缓存完成')
else:
    print('[缓存] 所有基金净值数据已在本地，跳过Wind')

# ------ 2. Duration 缓存检查（wss支持多代码批量，大幅减少API调用次数）------
rpt_date = wind_fetcher._get_latest_rpt_date(target_date)
missing_dur = [c for c in all_fund_codes
               if not cache.check_duration_exists(c, rpt_date)]

if missing_dur:
    print(f'[缓存] 需从Wind获取 {len(missing_dur)} 只基金季报久期（rptDate={rpt_date}）...')
    BATCH = 100  # wss 建议单次不超过100个代码
    fetched, failed = 0, 0
    for i in range(0, len(missing_dur), BATCH):
        batch = missing_dur[i:i + BATCH]
        codes_str = ','.join(batch)
        data = w.wss(codes_str, "risk_duration", f"rptDate={rpt_date}")
        if data.ErrorCode == 0 and data.Data and data.Data[0]:
            dur_dict = {}
            for code, val in zip(data.Codes, data.Data[0]):
                if val is not None and not (isinstance(val, float) and np.isnan(val)):
                    dur_dict[code] = float(val)
                else:
                    dur_dict[code] = None
            cache.write_duration_batch(rpt_date, dur_dict)
            fetched += len(batch)
        else:
            failed += len(batch)
    print(f'[缓存] 季报久期缓存完成（成功批次覆盖 {fetched} 只，失败批次 {failed} 只）')
else:
    print(f'[缓存] 所有基金季报久期已在本地（rptDate={rpt_date}），跳过Wind')

基金池共 1569 只
[缓存] 需从Wind获取 1569 只基金净值（共 1569 只）...
  净值进度: 20/1569
  净值进度: 40/1569
  净值进度: 60/1569
  净值进度: 80/1569
  净值进度: 100/1569
  净值进度: 120/1569
  净值进度: 140/1569
  净值进度: 160/1569
  净值进度: 180/1569
  净值进度: 200/1569
  净值进度: 220/1569
  净值进度: 240/1569
  净值进度: 260/1569
  净值进度: 280/1569
  净值进度: 300/1569
  净值进度: 320/1569
  净值进度: 340/1569
  净值进度: 360/1569
  净值进度: 380/1569
  净值进度: 400/1569
  净值进度: 420/1569
  净值进度: 440/1569
  净值进度: 460/1569
  净值进度: 480/1569
  净值进度: 500/1569
  净值进度: 520/1569
  净值进度: 540/1569
  净值进度: 560/1569
  净值进度: 580/1569
  净值进度: 600/1569
  净值进度: 620/1569
  净值进度: 640/1569
  净值进度: 660/1569
  净值进度: 680/1569
  净值进度: 700/1569
  净值进度: 720/1569
  净值进度: 740/1569
  净值进度: 760/1569
  净值进度: 780/1569
  净值进度: 800/1569
  净值进度: 820/1569
  净值进度: 840/1569
  净值进度: 860/1569
  净值进度: 880/1569
  净值进度: 900/1569
  净值进度: 920/1569
  净值进度: 940/1569
  净值进度: 960/1569
  净值进度: 980/1569
  净值进度: 1000/1569
  净值进度: 1020/1569
  净值进度: 1040/1569
  净值进度: 1060/1569
  净值进度: 1080/1569
  净值进度: 1100/1569
  净值进度: 1120/

## 4. 测试Wind数据获取

In [42]:
# # 测试获取单只基金数据
# test_code = '000033.OF'
# start_date = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')

# print(f'获取基金 {test_code} 从 {start_date} 到 {target_date} 的净值数据...')
# nav_df = wind_fetcher.get_fund_nav_smoothed(test_code, start_date, target_date)

# if nav_df is not None:
#     print(f'成功获取 {len(nav_df)} 条数据')
#     print(nav_df.head(10))

## 6. 创建久期计算器

In [43]:
# 创建久期计算器
calculator = FundDurationCalculator(
    data_preprocessor=preprocessor,
    fund_classifier=classifier,
    wind_fetcher=wind_fetcher,
    index_processor=index_processor
)

print('久期计算器创建成功')

久期计算器创建成功


## 7. 计算久期统计数据

In [44]:
# 计算久期统计数据 - 带详细进度输出
print(f'开始计算 {target_date} 的基金久期...')
print('='*60)

# 手动展开计算流程以便查看中间变量
import time

# 1. 获取基金池
print('\n【步骤1】获取基金池...')
fund_pool = calculator.data_preprocessor.get_fund_pool(target_date)
short_count = len(fund_pool.get('short', []))
ml_count = len(fund_pool.get('medium_long', []))
print(f'  短期纯债基金: {short_count} 只')
print(f'  中长期纯债基金: {ml_count} 只')
total_funds = short_count + ml_count
print(f'  总计: {total_funds} 只')

# 准备存储结果
results = {}
processed_count = 0
success_count = 0
error_codes = []
error_reasons = {}  # 统计错误原因
start_time = time.time()

# 2. 处理短期基金
print('\n【步骤2】处理短期纯债基金...')
print('-'*60)
fund_df_short = fund_pool.get('short', pd.DataFrame())

for idx, row in fund_df_short.iterrows():
    processed_count += 1
    fund_code = row['Code']
    fund_name = row['Name']
    
    # 前3只基金，每只都详细输出
    # 之后每10只基金输出一次进度
    show_detail = processed_count <= 3 or processed_count % 10 == 0 or processed_count == short_count
    
    if show_detail:
        elapsed = time.time() - start_time
        eta = elapsed / processed_count * (total_funds - processed_count) if processed_count > 0 else 0
        print(f'  [{processed_count}/{total_funds}] {fund_code} - {fund_name[:15]}')
    
    try:
        # 判断基金类型
        fund_bond_type = calculator.fund_classifier.get_fund_type(fund_code, target_date)
        
        if fund_bond_type == 'rate':
            index_codes = calculator.index_processor.short_rate_indices
        elif fund_bond_type == 'credit':
            index_codes = calculator.index_processor.short_credit_indices
        else:
            reason = '未知债券类型'
            error_codes.append((fund_code, fund_name, reason))
            error_reasons[reason] = error_reasons.get(reason, 0) + 1
            continue
        
        # 获取基金净值数据
        start_date_calc = (pd.to_datetime(target_date) - pd.Timedelta(days=60)).strftime('%Y-%m-%d')
        fund_nav_df = calculator.wind_fetcher.get_fund_nav_smoothed(
            fund_code, start_date_calc, target_date
        )
        
        if fund_nav_df is None:
            reason = '净值数据获取失败'
            error_codes.append((fund_code, fund_name, reason))
            error_reasons[reason] = error_reasons.get(reason, 0) + 1
            if show_detail:
                print(f'    ✗ {reason}')
            continue
        
        if show_detail:
            print(f'    ✓ 净值数据获取成功: {len(fund_nav_df)} 条')
        
        # 获取Wind披露久期（Lasso单因子退化时用于锚定额外因子）
        reported_duration = calculator.wind_fetcher.get_fund_reported_duration(
            fund_code, target_date
        )
        
        # 计算久期
        duration = calculator.duration_model.calculate_fund_duration(
            fund_nav_df, index_codes, target_date,
            reported_duration=reported_duration,
            fund_code=fund_code,
            method=weight_method
        )
        
        if duration is not None:
            results[fund_code] = {
                'duration': duration,
                'fund_type': 'short',
                'bond_type': fund_bond_type,
                'name': fund_name
            }
            success_count += 1
            if show_detail:
                print(f'    ✓ 久期: {duration:.3f} 年')
        else:
            reason = '久期计算失败'
            error_codes.append((fund_code, fund_name, reason))
            error_reasons[reason] = error_reasons.get(reason, 0) + 1
            if show_detail:
                print(f'    ✗ {reason}')
            
    except Exception as e:
        reason = f'异常: {str(e)[:60]}'
        error_codes.append((fund_code, fund_name, reason))
        error_reasons[reason] = error_reasons.get(reason, 0) + 1
        if show_detail:
            print(f'    ✗ {reason}')
        continue

print(f'\n短期基金处理完成: 成功 {success_count} 只')

# 3. 处理中长期基金
print('\n【步骤3】处理中长期纯债基金...')
print('-'*60)
fund_df_ml = fund_pool.get('medium_long', pd.DataFrame())
ml_start_count = success_count

for idx, row in fund_df_ml.iterrows():
    processed_count += 1
    fund_code = row['Code']
    fund_name = row['Name']
    
    # 每10只基金输出一次进度
    if processed_count % 10 == 0 or processed_count == total_funds:
        elapsed = time.time() - start_time
        eta = elapsed / processed_count * (total_funds - processed_count) if processed_count > 0 else 0
        ml_success = success_count - ml_start_count
        print(f'  进度: {processed_count}/{total_funds} | 成功: {success_count} (中长期:{ml_success}) | 耗时: {elapsed:.1f}s | 预计剩余: {eta:.1f}s')
    
    try:
        # 判断基金类型
        fund_bond_type = calculator.fund_classifier.get_fund_type(fund_code, target_date)
        
        if fund_bond_type == 'rate':
            index_codes = calculator.index_processor.medium_long_rate_indices
        elif fund_bond_type == 'credit':
            index_codes = calculator.index_processor.medium_long_credit_indices
        else:
            error_codes.append((fund_code, fund_name, '未知债券类型'))
            continue
        
        # 获取基金净值数据
        start_date_calc = (pd.to_datetime(target_date) - pd.Timedelta(days=90)).strftime('%Y-%m-%d')
        fund_nav_df = calculator.wind_fetcher.get_fund_nav_smoothed(
            fund_code, start_date_calc, target_date
        )
        
        if fund_nav_df is None:
            error_codes.append((fund_code, fund_name, '净值数据获取失败'))
            continue
        
        # 获取Wind披露久期（Lasso单因子退化时用于锚定额外因子）
        reported_duration = calculator.wind_fetcher.get_fund_reported_duration(
            fund_code, target_date
        )
        
        # 计算久期
        duration = calculator.duration_model.calculate_fund_duration(
            fund_nav_df, index_codes, target_date,
            reported_duration=reported_duration,
            fund_code=fund_code,
            method=weight_method
        )
        
        if duration is not None:
            results[fund_code] = {
                'duration': duration,
                'fund_type': 'medium_long',
                'bond_type': fund_bond_type,
                'name': fund_name
            }
            success_count += 1
            
    except Exception as e:
        error_codes.append((fund_code, fund_name, f'异常: {str(e)[:50]}'))
        continue

print(f'\n中长期基金处理完成: 成功 {success_count - ml_start_count} 只')

# 4. 汇总统计
print('\n【步骤4】汇总统计结果...')
print('='*60)
total_elapsed = time.time() - start_time
print(f'总耗时: {total_elapsed:.1f} 秒')
print(f'处理基金总数: {processed_count}')
print(f'成功计算久期: {success_count}')
print(f'失败数量: {len(error_codes)}')

# 显示失败原因统计
if error_reasons:
    print(f'\n失败原因统计:')
    for reason, count in sorted(error_reasons.items(), key=lambda x: -x[1]):
        print(f'  {reason}: {count} 只')

# 显示失败原因样例（前10个）
if error_codes:
    print(f'\n失败原因样例（前10个）:')
    for i, (code, name, reason) in enumerate(error_codes[:10]):
        print(f'  {i+1}. {code} - {name[:20]}: {reason}')

# 转换为DataFrame进行统计
if results:
    results_df = pd.DataFrame.from_dict(results, orient='index')
    print(f'\n成功计算的基金DataFrame形状: {results_df.shape}')
    print('前5只基金久期:')
    print(results_df[['name', 'duration', 'fund_type', 'bond_type']].head())
    
    # 分类统计
    stats = {}
    
    print('\n【步骤5】分类统计...')
    for fund_type in ['short', 'medium_long']:
        for bond_type in ['rate', 'credit']:
            key = f'{fund_type}_{bond_type}'
            
            mask = (results_df['fund_type'] == fund_type) & (results_df['bond_type'] == bond_type)
            subset = results_df[mask]
            
            if len(subset) > 0:
                durations = subset['duration'].values
                
                stats[key] = {
                    'count': len(durations),
                    'median': np.median(durations),
                    'mean': np.mean(durations),
                    'std': np.std(durations),
                    'cv': np.std(durations) / np.mean(durations) if np.mean(durations) > 0 else np.nan,
                    'min': np.min(durations),
                    'max': np.max(durations)
                }
                print(f'  {key}: {len(durations)} 只基金')
    
    # 显示最终结果
    if stats:
        print('\n' + '='*60)
        print('=== 久期统计结果 ===')
        for key, value in stats.items():
            print(f'\n{key}:')
            print(f'  基金数量: {value["count"]}')
            print(f'  中位数: {value["median"]:.3f} 年')
            print(f'  均值: {value["mean"]:.3f} 年')
            print(f'  标准差: {value["std"]:.3f}')
            print(f'  变异系数(分歧度): {value["cv"]:.3f}')
            print(f'  最小值: {value["min"]:.3f}')
            print(f'  最大值: {value["max"]:.3f}')
    else:
        print('未能计算出久期统计数据')
        stats = None
    
    # ===== 新增：剔除上界基金的统计 =====
    print('\n' + '='*60)
    print('=== 剔除上界后的统计 ===')
    
    # Identify funds that hit upper bound
    upper_bound_funds = set()
    for fund_code, boundary_type in calculator.duration_model.final_boundary_status.items():
        if boundary_type == 'upper':
            upper_bound_funds.add(fund_code)
    
    print(f'触发上界的基金数量: {len(upper_bound_funds)}')
    
    # Calculate statistics excluding upper-bound funds
    stats_excl_upper = {}
    
    print('\n【步骤5.1】剔除上界后的分类统计...')
    for fund_type in ['short', 'medium_long']:
        for bond_type in ['rate', 'credit']:
            key = f'{fund_type}_{bond_type}'
            
            mask = (results_df['fund_type'] == fund_type) & (results_df['bond_type'] == bond_type)
            subset = results_df[mask]
            
            # Exclude upper-bound funds
            subset_excl = subset[~subset.index.isin(upper_bound_funds)]
            
            if len(subset_excl) > 0:
                durations = subset_excl['duration'].values
                
                stats_excl_upper[key] = {
                    'count': len(durations),
                    'median': np.median(durations),
                    'mean': np.mean(durations),
                    'std': np.std(durations),
                    'cv': np.std(durations) / np.mean(durations) if np.mean(durations) > 0 else np.nan,
                    'min': np.min(durations),
                    'max': np.max(durations)
                }
                print(f'  {key}: {len(durations)} 只基金 (剔除 {len(subset) - len(subset_excl)} 只上界基金)')
    
    # Display comparison
    if stats_excl_upper:
        print('\n' + '='*60)
        print('=== 久期统计对比（全量 vs 剔除上界）===')
        for key in stats:
            print(f'\n{key}:')
            orig = stats[key]
            excl = stats_excl_upper.get(key)
            if excl:
                print(f'  全量: 中位数={orig["median"]:.3f}, 分歧度={orig["cv"]:.3f}, 样本={orig["count"]}')
                print(f'  剔除上界: 中位数={excl["median"]:.3f}, 分歧度={excl["cv"]:.3f}, 样本={excl["count"]}')
                diff = orig["median"] - excl["median"]
                print(f'  差异: {diff:+.3f} 年')
    
    # Update stats_final to include both sets
    stats_final = {'全量': stats, '剔除上界': stats_excl_upper}
else:
    print('未能计算出久期统计数据')
    stats_final = {'全量': None, '剔除上界': None}

print('\n' + '='*60)
print('计算完成！')

开始计算 2025-12-31 的基金久期...

【步骤1】获取基金池...
  短期纯债基金: 354 只
  中长期纯债基金: 1215 只
  总计: 1569 只

【步骤2】处理短期纯债基金...
------------------------------------------------------------
  [1/1569] 000084.OF - 博时安盈A
    ✓ 净值数据获取成功: 43 条
[边界] 000084.OF 2025-12-31 第1轮: 检测到upper边界，Σβ=1.4000，搜索最优swap
[无候选] 000084.OF 2025-12-31: upper界，池均久期=0.95Y，无满足方向约束的候选因子，当前解为最优
    ✓ 久期: 1.375 年
  [2/1569] 000089.OF - 民生加银高等级信用债C
    ✓ 净值数据获取成功: 43 条
[边界] 000089.OF 2025-12-31 第1轮: 检测到upper边界，Σβ=1.4000，搜索最优swap
[swap搜索] 000089.OF 2025-12-31: upper界，池均久期=0.75Y，候选因子1个: ['CBA01861.CS']
  [评估] 移除CBA01831.CS(dur=0.13Y) → 添加CBA01861.CS(dur=0.84Y): Q=1.04e-08, n=15
  [评估] 移除CBA01841.CS(dur=0.36Y) → 添加CBA01861.CS(dur=0.84Y): Q=3.84e-08, n=15
  [评估] 移除CBA01851.CS(dur=0.62Y) → 添加CBA01861.CS(dur=0.84Y): Q=3.98e-08, n=15
[最优swap] 000089.OF 2025-12-31: 移除CBA01831.CS → 添加CBA01861.CS, Q=1.04e-08
[调整] 000089.OF 2025-12-31 第1轮: 移除{CBA01831.CS}, 添加{CBA01861.CS}, 因子数4→4
    ✓ 久期: 0.721 年
  [3/1569] 000128.OF - 大成景安短融A
    ✓ 净值数据获取成功: 43 条
[边界

## 8. 保存结果

In [45]:
# 保存结果到Excel
if stats_final and stats_final.get('全量'):
    # Save both statistics sets to Excel with multiple sheets
    output_file = f'./output/久期统计结果_{weight_method}_{target_date.replace("-", "")}.xlsx'
    
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        # Save full statistics (全量)
        if stats_final['全量']:
            df_full = pd.DataFrame(stats_final['全量']).T
            df_full.to_excel(writer, sheet_name='全量统计', index=True)
        
        # Save statistics excluding upper-bound funds (剔除上界)
        if stats_final.get('剔除上界'):
            df_excl = pd.DataFrame(stats_final['剔除上界']).T
            df_excl.to_excel(writer, sheet_name='剔除上界统计', index=True)
    
    print(f'\n结果已保存到 {output_file}')
    print(f'  - Sheet "全量统计": 包含所有基金的统计')
    print(f'  - Sheet "剔除上界统计": 剔除触发上界基金的统计')
    
    # 同时保存详细结果
    detailed_results_file = f'./output/久期详细结果_{weight_method}_{target_date.replace("-", "")}.xlsx'
    results_df.to_excel(detailed_results_file)
    print(f'详细结果已保存到 {detailed_results_file}')
else:
    print('没有结果可保存')


结果已保存到 ./output/久期统计结果_linear_20251231.xlsx
  - Sheet "全量统计": 包含所有基金的统计
  - Sheet "剔除上界统计": 剔除触发上界基金的统计
详细结果已保存到 ./output/久期详细结果_linear_20251231.xlsx


## 9. 导出边界迭代日志

In [46]:
# 导出边界迭代日志到 Excel（需在 cell-11 主循环运行后执行）
log_path = calculator.export_iteration_logs(
    target_date=target_date,
    results_dict=results,          # 补充 fund_type / bond_type / duration
    output_path=f'./output/久期迭代日志_{target_date.replace("-", "")}.xlsx'
)


迭代日志已导出至: ./output/久期迭代日志_20251231.xlsx
  全量汇总: 1564 只基金
  触发边界: 1295 只基金
  迭代详情: 4368 条记录
